# 🎨 ComfyUI trên Google Colab (Free)

**Hướng dẫn:**
1. Vào menu `Runtime` → `Change runtime type` → chọn **T4 GPU** → Save
2. Chạy lần lượt **Cell 1 → Cell 2 → Cell 3** (bấm nút ▶ bên trái mỗi cell)
3. Ở Cell 3, đợi dòng link `https://xxxx.trycloudflare.com` hiện ra rồi bấm vào đó để mở giao diện ComfyUI

⚠️ Colab free sẽ mất toàn bộ file khi ngắt phiên — lần sau vào chạy lại từ Cell 1.

In [ ]:
# ===== CELL 1: Kiểm tra GPU + Cài ComfyUI =====
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os
os.chdir('/content')
!rm -rf /content/ComfyUI

# Clone ComfyUI (repo đang hoạt động bình thường, không phụ thuộc repo Stability-AI đã bị xóa)
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

os.chdir('/content/ComfyUI')
!pip install -q -r requirements.txt

print('\n✅ Cài ComfyUI xong! Chạy tiếp Cell 2.')

In [ ]:
# ===== CELL 2: Tải Model =====
# SD 1.5 bản fp16 (2.1GB) - mirror chính thức của Comfy-Org
# (link runwayml/stable-diffusion-v1-5 cũ đã bị gỡ khỏi Hugging Face)
!wget -c -O /content/ComfyUI/models/checkpoints/v1-5-pruned-emaonly-fp16.safetensors \
  "https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors"

# (TÙY CHỌN) Bỏ dấu # ở 2 dòng dưới nếu muốn thêm model ảnh người thật đẹp hơn:
#!wget -c -O /content/ComfyUI/models/checkpoints/RealisticVision51.safetensors \
#  "https://huggingface.co/SG161222/Realistic_Vision_V5.1_noVAE/resolve/main/Realistic_Vision_V5.1_fp16-no-ema.safetensors"

!ls -lh /content/ComfyUI/models/checkpoints/
print('\n✅ Tải model xong! Chạy tiếp Cell 3.')

In [ ]:
# ===== CELL 3: Khởi chạy ComfyUI + tạo link truy cập =====
# Cài cloudflared để tạo đường link công khai (thay cho --share của gradio)
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess, threading, time, socket, re

def open_tunnel(port):
    # Đợi ComfyUI mở cổng 8188 rồi mới tạo tunnel
    while True:
        time.sleep(1)
        try:
            with socket.create_connection(('127.0.0.1', port), timeout=1):
                break
        except OSError:
            pass
    p = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            print('\n' + '='*60)
            print('🎨 MỞ LINK NÀY ĐỂ DÙNG ComfyUI:', m.group(0))
            print('='*60 + '\n')
            break

threading.Thread(target=open_tunnel, args=(8188,), daemon=True).start()

import os
os.chdir('/content/ComfyUI')
!python main.py --dont-print-server

## 📝 Cách tạo ảnh đầu tiên

1. Mở link `trycloudflare.com` ở trên → giao diện ComfyUI hiện ra với workflow mặc định
2. Ô **Load Checkpoint**: chọn `v1-5-pruned-emaonly-fp16.safetensors`
3. Ô **CLIP Text Encode (Prompt)** phía trên: gõ mô tả ảnh bằng tiếng Anh, ví dụ:
   `a beautiful landscape in Vietnam, rice terraces, sunrise, highly detailed`
4. Ô prompt phía dưới (negative): gõ `blurry, low quality, ugly`
5. Bấm nút **Queue** → đợi vài giây → ảnh hiện ra ở ô **Save Image**

## ❓ Xử lý sự cố
- **Không thấy GPU ở Cell 1** → Runtime → Change runtime type → T4 GPU
- **Link tunnel không hiện** → chạy lại Cell 3
- **Hết phiên/mất kết nối** → Colab free giới hạn ~vài giờ GPU/ngày, chạy lại từ Cell 1